# Final Reproducible Experiment

**Uncertainty-Aware QUBO Framework for EV Charging Scheduling on Real Caltech ACN Data**

This is the **canonical reproducibility notebook** for the project. It orchestrates the existing tested pipeline (, , , , ) in the correct order, starting from environment validation and ending with final results, statistical validation, figures, reproducibility verification, and documented limitations.

## Scientific integrity

The methodology is **frozen**. No scientific parameter, scenario count, penalty, seed, optimizer setting, or shot count was tuned based on held-out results. The frozen configuration is  (version ), and its raw-file SHA-256 is:



This hash **must** match exactly. The notebook verifies it before any scientific execution.

The authoritative Stage 9 experiment used **real Caltech ACN data** (live API at , ). No synthetic substitution was used at any point in the authoritative execution.

## Execution modes

The notebook supports two explicit modes:

| Mode | Default | Behavior |
|---|---|---|
| **Artifact reproduction** | YES | Loads existing frozen Stage 9 artifacts and reproduces tables, figures, and statistical summaries. Does not fetch live data or rerun expensive QAOA. |
| **Full source reproduction** | NO | Performs Stage 5 live-data loading, regenerates scenarios, runs QAOA, executes Stage 9. **Requires**  and credentials. |

The default mode is **artifact reproduction**. The full source mode must be explicitly enabled by the user and is **strongly discouraged** unless the existing artifacts are missing or invalidated.


## 1. Environment Verification

Verify Python version, key dependencies, and repository structure. The notebook does **not** silently install missing packages.

In [ ]:
import sys, os, platform
print(f'Python version: {sys.version}')
print(f'Platform: {platform.platform()}')
print(f'Repository root: {os.getcwd()}')


In [ ]:
# Check key dependencies
missing = []
for mod_name, import_name in [
    ('numpy', 'numpy'),
    ('scipy', 'scipy'),
    ('qiskit', 'qiskit'),
    ('qiskit_aer', 'qiskit_aer'),
    ('matplotlib', 'matplotlib'),
]:
    try:
        mod = __import__(import_name)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'  [OK] {mod_name}: {ver}')
    except ImportError as e:
        print(f'  [MISSING] {mod_name}: {e}')
        missing.append(mod_name)
if missing:
    print()
    print(f'Missing dependencies: {missing}')
    print('These must be installed before running full reproduction mode.')
else:
    print()
    print('All required dependencies present.')


In [ ]:
# Verify repository structure
from pathlib import Path
ROOT = Path.cwd()
print(f'Repository root: {ROOT}')
print()
expected_paths = [
    'artifacts/final_experiment_config.json',
    'stage3/ev_scheduling.py',
    'stage4/qaoa.py',
    'stage5/uncertainty.py',
    'stage6/robust_qaoa.py',
    'stage9/real_experiment.py',
    'artifacts/stage10/FINAL_SCIENTIFIC_REPORT.md',
    'artifacts/stage10/STATISTICAL_DENOMINATOR_AUDIT.md',
    'artifacts/stage10/FINAL_RESULTS_TABLE.md',
    'artifacts/stage10/REPRODUCIBILITY_MANIFEST.json',
    'artifacts/stage10/figures/figure_1_heldout_performance.png',
    'artifacts/stage10/figures/figure_2_tradeoff.png',
    'artifacts/stage10/figures/figure_3_scenarios.png',
    'artifacts/stage10/figures/figure_4_distribution_shift.png',
    'artifacts/stage10/figures/figure_5_primary_ci.png',
]
all_present = True
for p in expected_paths:
    full = ROOT / p
    status = '[OK]' if full.exists() else '[MISSING]'
    if not full.exists():
        all_present = False
    print(f'  {status} {p}')
print()
print('All expected paths present:' if all_present else 'Some expected paths are missing.')


## 2. Frozen Configuration Verification

Load  and verify its raw-file SHA-256 matches the expected value. If the hash does not match, STOP.

**Frozen temporal windows:**
- Calibration: 2018-05-01 → 2019-07-01 (exclusive)
- Held-out: 2019-07-01 → 2020-01-01 (exclusive)


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_FROZEN_SHA256 = '4a08e1e65587cc904521ff1bfc955b30671a5cb3485664d8b75f1ad93d9013b3'
FROZEN_CONFIG_PATH = Path('artifacts/final_experiment_config.json')

raw = FROZEN_CONFIG_PATH.read_bytes()
computed_sha = hashlib.sha256(raw).hexdigest()
print(f'Frozen config path:        {FROZEN_CONFIG_PATH}')
print(f'Frozen config size (bytes): {len(raw)}')
print(f'Computed SHA-256:           {computed_sha}')
print(f'Expected SHA-256:           {EXPECTED_FROZEN_SHA256}')
match = computed_sha == EXPECTED_FROZEN_SHA256
print(f'Match: {match}')

if not match:
    print()
    print('=' * 70)
    print('FROZEN CONFIG INTEGRITY FAILURE')
    print('=' * 70)
    print('The frozen configuration file has been modified.')
    print('Scientific execution is FORBIDDEN.')
    print('Restore the file from a known-good backup or STOP.')
    raise SystemExit(1)
else:
    print()
    print('Frozen configuration integrity: VERIFIED')


In [ ]:
# Load the frozen config and display its scientific parameters
import json
with open('artifacts/final_experiment_config.json') as f:
    cfg = json.load(f)
print(f'Frozen config version: {cfg.get("version")}')
print()
print('Frozen methodology parameters:')
for k in ['K', 'alpha', 'rho_d', 'rho_p', 'rho_cap', 'M_window',
         'P_target_kW', 'P_site_max_kW', 'QAOA_p',
         'QAOA_optimizer', 'QAOA_seeds', 'QAOA_shots',
         'calibration_window_UTC', 'held_out_window_UTC']:
    v = cfg.get(k, '<not set>')
    print(f'  {k:25} = {v}')
print()
print('Frozen uncertainty definitions:')
for k, v in cfg.get('uncertainty_definitions', {}).items():
    print(f'  {k}: {v}')


## 3. Execution Mode

**This is the only cell you may need to modify.** The default mode is artifact reproduction, which loads existing frozen Stage 9 outputs and does not perform any new scientific computation.

| Mode | Value | Behavior |
|---|---|---|
| Artifact reproduction |  | Load existing  and  artifacts. Reproduce tables, figures, statistical summaries. **No API access, no QAOA, no overwrite of authoritative results.** |
| Full source reproduction |  | Call  (live API), run QAOA, execute . **Requires  env var, ~30 min wall time, will overwrite  and  artifacts.** |

**The default is .** Set to  only if you have explicitly authorized a full reproduction and have the credentials available.

In [ ]:
# User-controlled flag. Default is False (artifact reproduction).
RUN_FULL_EXPERIMENT = False

# Credential check (safe: presence only, never value).
import os
token_present = bool(os.environ.get('ACN_API_TOKEN') or os.environ.get('ACNPORTAL_TOKEN'))
print(f'RUN_FULL_EXPERIMENT = {RUN_FULL_EXPERIMENT}')
print(f'credential_available: {token_present}')
print()
if RUN_FULL_EXPERIMENT:
    if not token_present:
        print('WARNING: RUN_FULL_EXPERIMENT=True but no credential in environment.')
        print('Set ACN_API_TOKEN or ACNPORTAL_TOKEN before proceeding.')
    else:
        print('Full reproduction mode ACTIVE.')
        print('Will fetch live data and re-execute Stage 9. ~30 min wall time.')
        print('Will overwrite authoritative real_*.json and stage9_*.json artifacts.')
else:
    print('Artifact reproduction mode ACTIVE (default).')
    print('Will load existing frozen Stage 9 artifacts.')
    print('No API access, no QAOA, no overwrite.')


## 4. Repository Integrity

Verify the existence and SHA-256 of the source files implementing the pipeline. The implementation must not be modified between artifact reproduction runs.

In [ ]:
import hashlib
from pathlib import Path

SOURCE_FILES = {
    'Frozen config':           'artifacts/final_experiment_config.json',
    'Stage 3 ev_scheduling':   'stage3/ev_scheduling.py',
    'Stage 4 qaoa':            'stage4/qaoa.py',
    'Stage 5 uncertainty':     'stage5/uncertainty.py',
    'Stage 6 robust_qaoa':     'stage6/robust_qaoa.py',
    'Stage 9 driver':          'stage9/real_experiment.py',
}
for desc, p in SOURCE_FILES.items():
    full = Path(p)
    if full.exists():
        h = hashlib.sha256(full.read_bytes()).hexdigest()
        print(f'  [OK]   {desc:30}  {h}  {p}')
    else:
        print(f'  [MISSING] {desc:30}  {p}')
print()
print('Expected (from REPRODUCIBILITY_MANIFEST.json):')
import json
with open('artifacts/stage10/REPRODUCIBILITY_MANIFEST.json') as f:
    m = json.load(f)
print(f'  Frozen config: {m["frozen_configuration"]["sha256"]}')
for name, info in m['source_files'].items():
    print(f'  {name}: {info["sha256"]}')


## 5. Existing Artifact Validation

When , the notebook validates the existing Stage 9 artifacts. Required files:

- 
- 
- 
- , , 
- , 
- 
- 
- 
- 
- 
- 
- 
- 
- 
- 

In [ ]:
from pathlib import Path
import json

REQUIRED_ARTIFACTS = [
    'artifacts/real_raw_sessions.json',
    'artifacts/real_cleaning_results.json',
    'artifacts/real_uncertainty_statistics.json',
    'artifacts/real_scenarios_K4.json',
    'artifacts/real_scenarios_K8.json',
    'artifacts/real_scenarios_K16.json',
    'artifacts/real_calibration_parameters.json',
    'artifacts/real_calibration_freeze.json',
    'artifacts/real_f0_results.json',
    'artifacts/real_f1_results.json',
    'artifacts/real_f2_results.json',
    'artifacts/real_f3_results.json',
    'artifacts/real_qubo_validation.json',
    'artifacts/real_qaoa_results.json',
    'artifacts/real_heldout_results.json',
    'artifacts/real_paired_statistics.json',
    'artifacts/real_distribution_shift.json',
    'artifacts/real_objective_decomposition.json',
    'artifacts/real_failure_cases.json',
    'artifacts/stage9_audit.json',
    'artifacts/stage9_run.json',
]
all_present = True
for a in REQUIRED_ARTIFACTS:
    p = Path(a)
    status = '[OK]' if p.exists() else '[MISSING]'
    if not p.exists():
        all_present = False
    print(f'  {status} {a}')
print()
print('All required Stage 9 artifacts present:' if all_present else 'Some Stage 9 artifacts are MISSING.')
print()
if not all_present and not RUN_FULL_EXPERIMENT:
    print('To regenerate, set RUN_FULL_EXPERIMENT = True and re-run this notebook.')


In [ ]:
# Validate the run manifest's DATA_MODE and source
import json
with open('artifacts/stage9_run.json') as f:
    run = json.load(f)
print(f'Run manifest DATA_MODE: {run["DATA_MODE"]}')
print(f'Run manifest source:    {run["load_status"]["source"]}')
print(f'Run manifest base_url:  {run["load_status"]["base_url"]}')
print(f'Run manifest n_total:   {run["n_total"]}')
print(f'Run manifest n_cal:     {run["n_calibration"]}')
print(f'Run manifest n_ho:      {run["n_held_out"]}')
print()
# Confirm DATA_MODE = REAL and source = live_api
assert run['DATA_MODE'] == 'REAL', 'DATA_MODE is not REAL'
assert run['load_status']['source'] == 'live_api', 'source is not live_api'
print('DATA_MODE and source verified.')


## 6. Stage 5 Real Data Loading

**This section only executes when .**

The notebook imports the existing Stage 5 implementation. It does **not** duplicate the loader logic. Only sanitized aggregate information is reported.

In [ ]:
if RUN_FULL_EXPERIMENT:
    import os
    if not (os.environ.get('ACN_API_TOKEN') or os.environ.get('ACNPORTAL_TOKEN')):
        raise RuntimeError('Cannot run full experiment: ACN_API_TOKEN not in environment.')
    print('Loading real data from ACN API...')
    from stage5.uncertainty import load_real_uncertainty
    samples, load_status = load_real_uncertainty()
    n_total = load_status.get('n_records', 0)
    n_cal = sum(1 for s in samples if s.calibration)
    n_ho = sum(1 for s in samples if not s.calibration)
    n_valid_joint = sum(1 for s in samples if s.delta_d_minutes is not None and s.delta_e_kwh is not None)
    print(f'  n_records (in-window total): {n_total}')
    print(f'  n_calibration:               {n_cal}')
    print(f'  n_held_out:                  {n_ho}')
    print(f'  valid joint DeltaE + Deltad: {n_valid_joint}')
    print()
    print('Raw records and behavioral values are NOT printed.')
else:
    print('RUN_FULL_EXPERIMENT = False. Skipping Stage 5 real-data loading.')
    print('Using existing real_*.json artifacts for downstream analysis.')


## 7. Calibration Data and Scenario Generation

Use the existing frozen implementation. Report aggregate scenario diagnostics only.

In [ ]:
import json
with open('artifacts/real_scenarios_K8.json') as f:
    s = json.load(f)
print(f'K = {s["K"]}  (frozen: 8)')
print(f'Number of clusters: {len(s["scenarios"])}')
print(f'K-means seed (driver): 20260837  (frozen: 20260829 + K)')
print()
print('Cluster summary:')
print(f'{"cluster":>7}  {"n_obs":>6}  {"weight":>8}  {"Δd centroid (min)":>20}  {"ΔE centroid (kWh)":>20}')
for c in s['scenarios']:
    print(f'{c["cluster"]:>7}  {c["n_observations"]:>6}  {c["weight"]:>8.4f}  {c["centroid_delta_d_minutes"]:>20.2f}  {c["centroid_delta_e_kwh"]:>20.4f}')
print()
weights = [c['weight'] for c in s['scenarios']]
n_obs = [c['n_observations'] for c in s['scenarios']]
print(f'Sum of weights:     {sum(weights):.16f}  (== 1.0)')
print(f'Sum of n_observations: {sum(n_obs)}  (== 7492 calibration valid joint)')
print(f'All clusters non-empty: {all(c["n_observations"] > 0 for c in s["scenarios"])}')


In [ ]:
# Display the γ (ADOPT) calibration parameter
import json
with open('artifacts/real_calibration_parameters.json') as f:
    cp = json.load(f)
print('Calibration parameters (frozen before held-out use):')
print(f'  gamma (ADOPT):       {cp["gamma"]:.4f}')
print(f'  rho_d_robust:        {cp["rho_d_robust"]:.4f}')
print(f'  alpha:               {cp["alpha"]}  (frozen: 1.0)')
print(f'  M_window:            {int(cp["M_window"])}  (frozen: 1e6)')
print(f'  n_calibration_samples (window total): {cp["n_calibration_samples"]}')
print()
print(f'γ = 1 + α × mean(σᵢ / R̄ᵢ) = 1 + 1.0 × {cp["adopt_stats"]["mean_ratio_sigma_over_Rbar"]:.4f} = {cp["gamma"]:.4f}')
print(f'ρ_d_robust = γ × ρ_d = {cp["gamma"]:.4f} × 1.0 = {cp["rho_d_robust"]:.4f}')


## 8. QUBO Formulations

The four QUBO formulations are constructed by :

- **F0 (deterministic)**: No uncertainty. Frozen penalty weights (ρ_d, ρ_p, ρ_cap) = (1.0, 0.1, 0.5).
- **F1 (robust, K=8)**: Scenario-averaged QUBO over K=8 empirical scenarios from calibration. Same penalty weights as F0.
- **F2 (ADOPT)**: Same scenario-averaged QUBO but with ρ_d scaled by γ = 4.1313 (so effective ρ_d = 4.1313).
- **F3 (oracle, analysis-only)**: QUBO built using the held-out mean (Δd, ΔE) as the realized scenario. **Not a deployable method**; included only as an upper-bound reference.

In [ ]:
import json
print('QUBO formulations (from real_*.json):')
print()
for tag, desc in [('f0', 'F0_deterministic'),
                  ('f1', 'F1_robust (K=8)'),
                  ('f2', 'F2_adopt (γ=4.1313)'),
                  ('f3', 'F3_oracle_analysis_only')]:
    with open(f'artifacts/real_{tag}_results.json') as f:
        r = json.load(f)
    print(f'  {tag.upper()}: {r["method"]}')
    print(f'    classical_optimum: {r["classical_optimum"]}')
    print(f'    n_vars: {len(r["Q_diagonal"])}  (expected 11)')
    print(f'    is_pure_qubo: {r["is_pure_qubo"]}')
    print(f'    n_auxiliary_vars: {r["n_auxiliary_vars"]}')
    print(f'    Q_off_diagonal_count: {r["Q_off_diagonal_count"]}')
    print()
# Validate QUBO structure
with open('artifacts/real_qubo_validation.json') as f:
    v = json.load(f)
print(f'Corrected-robust-QUBO algebraic validation:')
print(f'  n_bitstrings: {v["n_bitstrings"]}  (== 2^11)')
print(f'  mean_offset: {v["mean_offset"]:.2e}')
print(f'  max_abs_deviation: {v["max_abs_deviation_from_mean"]:.2e}')
print(f'  tolerance: {v["tolerance"]}')
print(f'  passes: {v["passes"]}')


## 9. QAOA

The frozen QAOA configuration:

- Depth p = 1
- Optimizer = COBYLA (via , maxiter=30, tol=1e-4, rhobeg=0.05, catol=0.002)
- Seeds = [0, 1, 2]
- Shots = 1024
- Backend =  (local simulator)
- Transpilation = , , 

QAOA is executed for F0, F1, F2 (not F3, which is analysis-only). Each (formulation, seed) combination produces a distinct per-seed record.

In [ ]:
import json
with open('artifacts/real_qaoa_results.json') as f:
    q = json.load(f)
print('QAOA per-formulation, per-seed results:')
print()
for fname in ['F0', 'F1', 'F2']:
    r = q[fname]
    print(f'{fname}:')
    print(f'  p:       {r["p"]}  (frozen: 1)')
    print(f'  shots:   {r["shots"]}  (frozen: 1024)')
    print(f'  n_seeds: {r["n_seeds"]}  (frozen: 3)')
    seeds = [s["seed"] for s in r['per_seed']]
    print(f'  seeds:   {seeds}  (frozen: [0,1,2])')
    print(f'  n_qubits: {r["n_qubits"]}  (expected 11)')
    print(f'  classical_optimum: {r["classical_optimum"]}')
    for s in r['per_seed']:
        print(f'    seed {s["seed"]}: AR={s["approximation_ratio"]:.4f}, P_feas={s["P_feasible"]:.4f}, runtime={s["runtime_s"]:.1f}s')
    ar_med = r['AR_median']
    ar_mean = r['AR_mean']
    pfe_mean = r['P_feasible_mean']
    pfe_std = r['P_feasible_std']
    print(f'  AR_median: {ar_med:.6f}, AR_mean: {ar_mean:.6f}')
    print(f'  P_feasible_mean: {pfe_mean:.4f} ± {pfe_std:.4f}')
    print()


## 10. Held-Out Evaluation

Each formulation's classical optimum schedule is applied to each held-out session's realized (Δd, ΔE). The headline P(feasible) uses the **valid joint held-out sample count (4,076)** as the denominator.

In [ ]:
import json
with open('artifacts/real_heldout_results.json') as f:
    h = json.load(f)
print('Held-out evaluation (denominator = 4,076 valid joint sessions):')
print()
print(f'{"Formulation":<30} {"P_feasible":>10} {"mean_unmet":>12} {"p95_unmet":>10} {"deadline_viol":>14} {"site_viol":>10}')
print('-' * 100)
for fname in ['F0', 'F1', 'F2', 'F3_oracle_analysis_only']:
    r = h[fname]
    print(f'{fname:<30} {r["P_feasible"]:>10.4f} {r["mean_unmet_kWh"]:>12.4f} {r["p95_unmet_kWh"]:>10.4f} {r["deadline_violation_rate"]:>14.4f} {r["site_violation_rate"]:>10.4f}')
print()
# Validate denominator
for fname in ['F0', 'F1', 'F2']:
    n = h[fname]['n_held_out_total']
    pf = h[fname]['P_feasible']
    nf = h[fname]['n_feasible']
    assert abs(pf - nf / max(1, n)) < 1e-9, f'{fname}: P_feasible inconsistent with n_feasible/n_held_out_total'
print('P_feasible denominators verified: n_feasible / n_held_out_total matches P_feasible for all F0/F1/F2.')


## 11. Statistical Validation

**Denominator convention (known issue, fully documented in ):**

The bootstrap operates over a vector of length **4,857** (the full held-out list, including 781 null-userInputs records represented as default-zero placeholders). The headline P(feasible uses n=**4,076** (valid joint only). The bootstrap's diff_mean and CI are internally consistent under the 4,857-vector convention; the difference is scaled by 4076/4857 = 0.8391 relative to the headline P_feasible difference. The Stage 10 forensic audit classified this as a **STATISTICAL REPORTING ISSUE REQUIRING QUALIFICATION** — not a scientific validity issue. The statistical conclusion (F2 < F0 on P(feasible)) is robust to denominator choice because both the reported CI and the implied valid-joint-only CI exclude 0.

This issue is **not** silently papered over.

In [ ]:
import json
with open('artifacts/real_paired_statistics.json') as f:
    p = json.load(f)
print('Paired bootstrap 95% CIs:')
print()
print(f'n_bootstrap:    {p["n_bootstrap"]}  (frozen: 10000)')
print(f'bonferroni_alpha: {p["bonferroni_alpha"]:.6f}  (== 0.05/3 = {0.05/3:.6f})')
print()
for label in ['primary_F2_vs_F0', 'secondary_F1_vs_F0', 'secondary_F2_vs_F1']:
    s = p[label]
    valid_ci = s['ci_lo'] <= s['diff_mean'] <= s['ci_hi']
    excludes_zero = s['ci_lo'] > 0 or s['ci_hi'] < 0
    print(f'{label}:')
    print(f'  n: {s["n"]}  (note: n=4,857 = full ho list length)')
    print(f'  diff_mean: {s["diff_mean"]:.6f}')
    print(f'  95% CI: [{s["ci_lo"]:.6f}, {s["ci_hi"]:.6f}]')
    print(f'  mean_a: {s["mean_a"]:.4f}  mean_b: {s["mean_b"]:.4f}')
    print(f'  CI ordering valid: {valid_ci}')
    print(f'  CI excludes 0: {excludes_zero}')
    print()
print('=' * 70)
print('DENOMINATOR QUALIFICATION (per STATISTICAL_DENOMINATOR_AUDIT.md):')
print('=' * 70)
print('  bootstrap n=4,857 includes 781 null-userInputs positions encoded as 0')
print('  headline P_feasible denominator=4,076 (valid joint only)')
print('  bootstrap diff_mean is (4076/4857) × headline_diff')
print('  Implied valid-joint-only diff_mean = -0.1011 × (4857/4076) = -0.1205')
print('  Implied valid-joint-only CI = 1.192 × reported CI = [-0.1305, -0.1104]')
print('  Both CIs exclude 0. Conclusion robust to denominator choice.')
print('  Classification: STATISTICAL REPORTING ISSUE REQUIRING QUALIFICATION.')


## 12. Final Results Table

Publication-ready table derived from the authoritative . The denominator for P(feasible) is the **valid joint held-out sample count (n=4,076)**.

In [ ]:
import json
print('=' * 95)
print('FINAL RESULTS TABLE — Held-out evaluation on 4,076 valid joint behavioral sessions')
print('DATA_MODE: REAL  |  Source: live ACN-Data API  |  Frozen methodology v1.0')
print('=' * 95)
with open('artifacts/real_heldout_results.json') as f:
    h = json.load(f)
print()
print(f'{"Method":<28} {"P_feasible":>10} {"n_feasible":>10} {"mean_unmet":>12} {"p95_unmet":>10} {"deadline_viol":>14} {"site_viol":>10}')
print('-' * 95)
for fname, label in [('F0', 'F0 (deterministic)'),
                      ('F1', 'F1 (robust K=8)'),
                      ('F2', 'F2 (ADOPT)'),
                      ('F3_oracle_analysis_only', 'F3 (oracle, analysis-only)')]:
    r = h[fname]
    print(f'{label:<28} {r["P_feasible"]:>10.4f} {r["n_feasible"]:>10} {r["mean_unmet_kWh"]:>12.4f} {r["p95_unmet_kWh"]:>10.4f} {r["deadline_violation_rate"]:>14.4f} {r["site_violation_rate"]:>10.4f}')
print()
print('Primary statistical endpoint:')
with open('artifacts/real_paired_statistics.json') as f:
    p = json.load(f)
s = p['primary_F2_vs_F0']
print(f'  F2 vs F0: diff_mean = {s["diff_mean"]:.4f}, 95% CI = [{s["ci_lo"]:.4f}, {s["ci_hi"]:.4f}]')
print(f'  CI excludes 0: {s["ci_lo"] > 0 or s["ci_hi"] < 0}  (statistically significant)')
print(f'  Sign: {"F2 < F0 (lower feasibility)" if s["diff_mean"] < 0 else "F2 > F0 (higher feasibility)"}')


## 13. Figures

The five Stage 10 figures are loaded from the authoritative directory and displayed inline.

In [ ]:
from IPython.display import Image, display
import os
figures = [
    ('artifacts/stage10/figures/figure_1_heldout_performance.png', 'Figure 1: Held-out performance comparison (F0/F1/F2/F3)'),
    ('artifacts/stage10/figures/figure_2_tradeoff.png', 'Figure 2: Pareto trade-off: feasibility vs unmet energy'),
    ('artifacts/stage10/figures/figure_3_scenarios.png', 'Figure 3: K=8 calibration scenario centroids'),
    ('artifacts/stage10/figures/figure_4_distribution_shift.png', 'Figure 4: Distribution shift diagnostic (cal vs ho)'),
    ('artifacts/stage10/figures/figure_5_primary_ci.png', 'Figure 5: Paired bootstrap 95% CIs'),
]
for path, caption in figures:
    if os.path.exists(path):
        print(caption)
        display(Image(filename=path))
        print()
    else:
        print(f'MISSING: {path}')


## 14. Reproducibility Manifest

Load and display the relevant reproducibility information from the Stage 10 manifest.

In [ ]:
import json
with open('artifacts/stage10/REPRODUCIBILITY_MANIFEST.json') as f:
    m = json.load(f)
print(f'DATA_MODE: {m["project"]["data_mode"]}')
print(f'Source:    {m["project"]["source"]}')
print()
print('Frozen configuration:')
fc = m['frozen_configuration']
print(f'  Path:        {fc["path"]}')
print(f'  SHA-256:     {fc["sha256"]}')
print(f'  Expected:    {fc["expected_sha256"]}')
print(f'  Verified:    {fc["verified_unchanged"]}')
print(f'  Version:     {fc["version"]}')
print()
print('Source file SHA-256:')
for name, info in m['source_files'].items():
    print(f'  {name}: {info["sha256"]}')
    print(f'    modification: {info["modification"]}')
print()
print('Sanitization status:')
for k, v in m['sanitization'].items():
    if k != 'verification_method':
        print(f'  {k}: {v}')
print()
print(f'Final status (per Stage 10): {m["final_status"]}')


## 15. Scientific Interpretation

**The proposed methods F1/F2 did NOT improve held-out feasibility relative to F0.** The primary result is a trade-off, not dominance.

- **F0 (deterministic)** achieves the highest held-out P(feasible) at 99.85%, with the lowest deadline violation rate (0.15%). It over-delivers energy to ensure deadline compliance (mean unmet 2.46 kWh).
- **F1 / F2 (scenario-robust / ADOPT)** trade 12.19 percentage points of feasibility for 1.71 kWh of unmet-demand reduction. They produce **identical schedules** on the 11-qubit  instance.
- The primary statistical endpoint (F2 vs F0 P(feasible), paired bootstrap 95% CI [−0.1095, −0.0926]) is **statistically significant** but the sign is **unfavorable** to the proposed method: F0 has higher feasibility than F2 on held-out data.

The proposed method **does not** dominate the deterministic baseline on the Pareto frontier. It is a **Pareto trade-off** with the F0 baseline. Whether this trade-off is operationally acceptable depends on deployment-specific priorities (cost/peak/unmet vs feasibility/deadline) that are outside the scope of this evaluation.

## 16. Limitations

1. **toy_B_3x4 instance scale**: 3 EVs, 4 slots, 11 qubits. The ADOPT γ-inflation does not change the discrete classical optimum on this small instance.
2. **F1 = F2 schedule identity**: On this instance, F1 and F2 produce identical schedules. The ADOPT framework's distinct contribution cannot be evaluated at this scale.
3. **Behavioral-data availability asymmetry**: 63.98% of calibration records have null  vs 16.08% in held-out (historical rollout of the Caltech ACN user-input feature).
4. **Bootstrap denominator convention**: The bootstrap operates over a 4,857-vector with default-zero placeholders; the conclusion is robust to denominator choice, but the convention is non-standard. See .
5. **Distribution shift**: A_mild, but the held-out ΔE range (max +3.4 kWh) is a strict subset of the calibration range (max +69.5 kWh). Worst-case performance may be understated.
6. **Real-world generalization**: Results are specific to the Caltech ACN deployment.
7. **QAOA at p=1**: Frozen p=1 is a weak variational form. Higher p is not in the frozen configuration.
8. **No demonstrated quantum advantage**: The QAOA at p=1 with COBYLA achieves AR=1.0 on F0/F1 but AR>1 on F2 seed 2 (sampling noise, faithfully reported).
9. **Scientific hypothesis NOT supported**: The proposed method did not improve on the deterministic baseline on this instance. The pipeline executed successfully, but the science does not support the hypothesis.

## 17. Final Conclusion

The Stage 9 real-data experiment **successfully executed** end-to-end on the real Caltech ACN dataset using the frozen methodology. The pipeline is reproducible, all 22 Stage 9 artifacts are present and internally consistent, no synthetic substitution was used, no credentials were leaked, no scientific parameters were tuned based on held-out results.

**The scientific conclusion is honest and unfavorable to the proposed method:**

The scenario-robust (F1) and ADOPT (F2) formulations did **not** improve held-out feasibility over the deterministic baseline (F0) on the  instance. F0 achieves 99.85% P(feasible) vs F1/F2 at 87.81%. The primary statistical endpoint is significant (CI excludes 0, p ≪ 0.0001) but the sign is **unfavorable** to F2.

**Final project status (consistent with Stage 10):**

**COMPLETE — WITH DOCUMENTED LIMITATIONS**

The proposed method achieves a Pareto trade-off, not dominance. The framework is reproducible; the methodology is frozen; the data is real; the conclusion is honest.